![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [1]:
import pickle
import pandas as pd

# Load the dataframe
merged_common_final_selected = pd.read_pickle('merged_common_final_selected.pkl')

# Load the feature list
with open('final_features.pkl', 'rb') as f:
    final_features = pickle.load(f)

# Load target_cols
with open('target_cols.pkl', 'rb') as f:
    target_cols = pickle.load(f)
    
print(f"✓ Loaded dataframe: {merged_common_final_selected.shape}")
print(f"✓ Loaded features: {len(final_features)} features")

In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, matthews_corrcoef, f1_score, precision_score, recall_score
import pandas as pd
import numpy as np

# ===== DECISION TREE TRAINING: ALL 6 TARGETS =====
print("\n" + "="*80)
print("DECISION TREE TRAINING: ALL 6 TARGETS WITH 14 CONSENSUS FEATURES")
print("="*80)

X_all = merged_common_final_selected[final_features]

print(f"\nDataset: {len(X_all):,} rows × {len(final_features)} features")

# Fill NaNs with -999
X_all_filled = X_all.fillna(-999)

# Store results
tree_results = {}
all_trees = {}

from sklearn.model_selection import TimeSeriesSplit

for target_name in target_cols:
    y_all = merged_common_final_selected[target_name]
    
    # TIME-SERIES SPLIT (respects temporal order + tests multiple regimes)
    tscv = TimeSeriesSplit(n_splits=5, test_size=int(len(X_all_filled) * 0.15))
    
    fold_results = []
    
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_all_filled)):
        X_train, X_test = X_all_filled.iloc[train_idx], X_all_filled.iloc[test_idx]
        y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
        
        # Calculate class weight for imbalance
        n_neg = (y_train == 0).sum()
        n_pos = (y_train == 1).sum()
        scale_pos_weight = n_neg / n_pos
        class_weight_dict = {0: 1.0, 1: scale_pos_weight}
        
        # Train decision tree
        tree = DecisionTreeClassifier(
            max_depth=3,
            min_samples_split=100,
            min_samples_leaf=50,
            min_impurity_decrease=0.0001,
            max_features='sqrt',
            class_weight=class_weight_dict,
            random_state=42
        )
        
        tree.fit(X_train, y_train)
        
        # Predictions
        test_pred = tree.predict(X_test)
        test_pred_proba = tree.predict_proba(X_test)[:, 1]
        
        # Metrics
        fold_results.append({
            'fold': fold_idx,
            'test_acc': accuracy_score(y_test, test_pred),
            'roc_auc': roc_auc_score(y_test, test_pred_proba),
            'mcc': matthews_corrcoef(y_test, test_pred),
            'f1': f1_score(y_test, test_pred)
        })
    
    # Aggregate across folds
    fold_df = pd.DataFrame(fold_results)
    tree_results[target_name] = {
        'pos_rate': y_all.mean(),
        'imbalance_ratio': scale_pos_weight,
        'test_acc': fold_df['test_acc'].mean(),
        'test_acc_std': fold_df['test_acc'].std(),
        'roc_auc': fold_df['roc_auc'].mean(),
        'mcc': fold_df['mcc'].mean(),
        'f1': fold_df['f1'].mean()
    }
    
    # Train final model on all data for rule extraction
    n_neg = (y_all == 0).sum()
    n_pos = (y_all == 1).sum()
    class_weight_dict = {0: 1.0, 1: n_neg/n_pos}
    
    final_tree = DecisionTreeClassifier(
        max_depth=3,
        min_samples_split=100,
        min_samples_leaf=50,
        min_impurity_decrease=0.0001,
        max_features='sqrt',
        class_weight=class_weight_dict,
        random_state=42
    )
    final_tree.fit(X_all_filled, y_all)
    all_trees[target_name] = final_tree

# Summary
print("\n" + "="*80)
print("SUMMARY: DECISION TREES (MAX DEPTH=5)")
print("="*80)

summary_df = pd.DataFrame(tree_results).T
print(summary_df.round(4).to_string())

print("\n" + "="*80)
print("AVERAGES")
print("="*80)
print(f"Avg ROC-AUC:     {summary_df['roc_auc'].mean():.4f}")
print(f"Avg F1:          {summary_df['f1'].mean():.4f}")
print(f"Avg MCC:         {summary_df['mcc'].mean():.4f}")

print("\n✓ Decision tree training complete!")

In [ ]:
from sklearn.tree import _tree
import numpy as np
import pandas as pd

# ===== EXTRACT RULES FROM BEST TARGET =====
print("\n" + "="*80)
print("EXTRACTING RULES FROM TREE: target_n3_001")
print("="*80)

# Get the tree for best target
tree = all_trees['target_n3_001']
target_name = 'target_n3_001'

# Recreate train/test split for this target
y_all = merged_common_final_selected[target_name]
X_all_filled = X_all.fillna(-999)

split_idx = int(len(X_all) * 0.7)
X_train_filled = X_all_filled.iloc[:split_idx]
X_test_filled = X_all_filled.iloc[split_idx:]
y_train = y_all.iloc[:split_idx]
y_test = y_all.iloc[split_idx:]

def extract_rules(tree, feature_names):
    """Extract all decision rules from tree"""
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!" 
                    for i in tree_.feature]
    
    paths = []
    
    def recurse(node, path):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            # Internal node
            name = feature_name[node]
            threshold = tree_.threshold[node]
            
            # Left branch
            recurse(tree_.children_left[node], path + [(name, '<=', threshold)])
            # Right branch
            recurse(tree_.children_right[node], path + [(name, '>', threshold)])
        else:
            # Leaf node
            samples = tree_.n_node_samples[node]
            value_counts = tree_.value[node][0, :] * samples
            
            n_down = int(value_counts[0])
            n_up = int(value_counts[1])
            class_pred = np.argmax(value_counts)
            confidence = value_counts[class_pred] / samples
            
            paths.append({
                'rule': path,
                'prediction': 'UP' if class_pred == 1 else 'DOWN',
                'samples': samples,
                'confidence': confidence,
                'n_down': n_down,
                'n_up': n_up
            })
    
    recurse(0, [])
    return paths

# Extract rules
rules = extract_rules(tree, X_train_filled.columns.tolist())

# Evaluate each rule on train AND test set
rule_performance = []

for idx, rule_info in enumerate(rules):
    conditions = rule_info['rule']
    
    # Apply conditions on TRAIN set
    train_mask = np.ones(len(X_train_filled), dtype=bool)
    for feature, operator, threshold in conditions:
        if operator == '<=':
            train_mask &= (X_train_filled[feature] <= threshold).values
        else:
            train_mask &= (X_train_filled[feature] > threshold).values
    
    n_train_matched = train_mask.sum()
    pred_class = 1 if rule_info['prediction'] == 'UP' else 0
    
    if n_train_matched > 0:
        train_accuracy = (y_train[train_mask] == pred_class).mean()
    else:
        train_accuracy = 0
    
    # Apply conditions on TEST set
    test_mask = np.ones(len(X_test_filled), dtype=bool)
    for feature, operator, threshold in conditions:
        if operator == '<=':
            test_mask &= (X_test_filled[feature] <= threshold).values
        else:
            test_mask &= (X_test_filled[feature] > threshold).values
    
    n_test_matched = test_mask.sum()
    
    if n_test_matched > 0:
        test_accuracy = (y_test[test_mask] == pred_class).mean()
    else:
        test_accuracy = 0
    
    rule_performance.append({
        'rule_id': idx,
        'prediction': rule_info['prediction'],
        'train_samples': n_train_matched,
        'train_accuracy': train_accuracy,
        'test_samples': n_test_matched,
        'test_accuracy': test_accuracy,
        'train_test_gap': abs(train_accuracy - test_accuracy),
        'conditions': conditions
    })

# Create DataFrame and filter
rules_df = pd.DataFrame(rule_performance)

# Filter: require minimum samples and reasonable performance
rules_df_filtered = rules_df[
    (rules_df['test_samples'] > 0) &  # At least 10 test samples
    (rules_df['train_samples'] > 00) &  # At least 20 train samples
    (rules_df['train_accuracy'] > 0.51)  # Train accuracy above random
].copy()

print("\n" + "="*80)
print("TOP 10 RULES BY TEST ACCURACY (with low train-test gap)")
print("="*80)

# Sort by test accuracy, then by smallest gap
rules_df_filtered['score'] = rules_df_filtered['test_accuracy'] - (0.5 * rules_df_filtered['train_test_gap'])
rules_by_test = rules_df_filtered.sort_values('score', ascending=False)

for idx, row in rules_by_test.head(10).iterrows():
    print(f"\n{'='*60}")
    print(f"Rule {row['rule_id']}: Predict {row['prediction']}")
    print(f"{'='*60}")
    print(f"Train: {row['train_samples']} samples | Accuracy: {row['train_accuracy']:.1%}")
    print(f"Test:  {row['test_samples']} samples | Accuracy: {row['test_accuracy']:.1%}")
    print(f"Train-Test Gap: {row['train_test_gap']:.1%} {'✓ STABLE' if row['train_test_gap'] < 0.10 else '⚠️ UNSTABLE'}")
    print(f"\nConditions (IF all are true, THEN predict {row['prediction']}):")
    for i, (feature, op, threshold) in enumerate(row['conditions'], 1):
        if threshold == -999:
            print(f"  {i}. {feature} {op} -999 (missing data)")
        else:
            print(f"  {i}. {feature} {op} {threshold:.6f}")

print("\n" + "="*80)
print("SUMMARY & INTERPRETATION")
print("="*80)
print(f"Total rules extracted: {len(rules_df)}")
print(f"Usable rules (filtered): {len(rules_df_filtered)}")
print(f"Best test accuracy: {rules_df_filtered['test_accuracy'].max():.1%}")
print(f"Mean test accuracy: {rules_df_filtered['test_accuracy'].mean():.1%}")
print(f"Mean train-test gap: {rules_df_filtered['train_test_gap'].mean():.1%}")
print(f"Most stable rule gap: {rules_df_filtered['train_test_gap'].min():.1%}")

# Flag suspicious rules
suspicious = rules_df_filtered[rules_df_filtered['train_test_gap'] > 0.20]
print(f"\n⚠️ Suspicious rules (gap > 20%): {len(suspicious)}")
if len(suspicious) > 0:
    print("   These rules likely overfit or got lucky on test set")

In [14]:
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# ===== PARAMETERS =====
N_TREES = 5
TREE_DEPTHS = [2, 3]
MIN_TEST_SAMPLES = 50
MIN_TEST_PCT = 1.0
NUMBER_OF_RULES = 300
POSITIVE_ONLY = True

print("="*80)
print("ENSEMBLE RULE EXTRACTION ACROSS ALL TARGETS")
print("="*80)
print(f"Trees per target: {N_TREES}")
print(f"Tree depths: {TREE_DEPTHS}")
print(f"Filters: ≥{MIN_TEST_SAMPLES} samples, ≥{MIN_TEST_PCT}% coverage")
print(f"Show top: {NUMBER_OF_RULES} rules")
print(f"Positive only: {POSITIVE_ONLY}")

# Use the same features and filled data from your tree training
X_all_filled = merged_common_final_selected[final_features].fillna(-999)

def extract_rules(tree, feature_names):
    """Extract all decision rules from tree"""
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!" 
                    for i in tree_.feature]
    
    paths = []
    
    def recurse(node, path):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            recurse(tree_.children_left[node], path + [(name, '<=', threshold)])
            recurse(tree_.children_right[node], path + [(name, '>', threshold)])
        else:
            samples = tree_.n_node_samples[node]
            weighted_values = tree_.value[node][0]
            class_pred = int(np.argmax(weighted_values))
            total_weight = weighted_values.sum()
            confidence = weighted_values[class_pred] / total_weight if total_weight > 0 else 0
            
            paths.append({
                'rule': path,
                'prediction': class_pred,
                'samples': samples,
                'confidence': confidence,
            })
    
    recurse(0, [])
    return paths

# Store all rules across all targets and depths
all_rules_global = []

for depth in TREE_DEPTHS:
    for target_name in target_cols:
        y_all = merged_common_final_selected[target_name]
        
        # Split data (70/30)
        split_idx = int(len(X_all_filled) * 0.7)
        X_train = X_all_filled.iloc[:split_idx]
        X_test = X_all_filled.iloc[split_idx:]
        y_train = y_all.iloc[:split_idx]
        y_test = y_all.iloc[split_idx:]
        
        total_test = len(X_test)
        
        # Calculate class weights
        n_neg = (y_train == 0).sum()
        n_pos = (y_train == 1).sum()
        scale_pos_weight = n_neg / n_pos
        class_weight_dict = {0: 1.0, 1: scale_pos_weight}
        
        # Train multiple trees
        for i in range(N_TREES):
            tree = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=100,
                min_samples_leaf=50,
                min_impurity_decrease=0.0001,
                max_features='sqrt',
                class_weight=class_weight_dict,
                random_state=42+i
            )
            
            # Bootstrap sample
            sample_idx = np.random.RandomState(42+i).choice(
                len(X_train), 
                size=int(len(X_train)*0.8), 
                replace=True
            )
            X_sample = X_train.iloc[sample_idx]
            y_sample = y_train.iloc[sample_idx]
            
            tree.fit(X_sample, y_sample)
            
            # Extract rules
            rules = extract_rules(tree, X_train.columns.tolist())
            
            # Evaluate each rule on test set
            for rule_info in rules:
                conditions = rule_info['rule']
                
                # Apply to test
                test_mask = np.ones(len(X_test), dtype=bool)
                for feature, operator, threshold in conditions:
                    if operator == '<=':
                        test_mask &= (X_test[feature] <= threshold).values
                    else:
                        test_mask &= (X_test[feature] > threshold).values
                
                test_matched = test_mask.sum()
                test_pct = (test_matched / total_test) * 100
                
                # Apply filters
                if test_matched >= MIN_TEST_SAMPLES and test_pct >= MIN_TEST_PCT:
                    test_accuracy = (y_test[test_mask] == rule_info['prediction']).mean()
                    
                    all_rules_global.append({
                        'target': target_name,
                        'depth': depth,
                        'tree_id': i,
                        'prediction': 'POSITIVE' if rule_info['prediction'] == 1 else 'NEGATIVE',
                        'train_samples': rule_info['samples'],
                        'train_confidence': rule_info['confidence'],
                        'test_samples': test_matched,
                        'test_pct': test_pct,
                        'test_accuracy': test_accuracy,
                        'n_conditions': len(conditions),
                        'conditions': conditions
                    })

# Create DataFrame and sort
all_rules_df = pd.DataFrame(all_rules_global)

if len(all_rules_df) == 0:
    print("\n⚠ NO RULES FOUND - try lowering MIN_TEST_SAMPLES or MIN_TEST_PCT")
else:
    # Filter by POSITIVE_ONLY if enabled
    if POSITIVE_ONLY:
        all_rules_df = all_rules_df[all_rules_df['prediction'] == 'POSITIVE']
    
    all_rules_df = all_rules_df.sort_values('test_accuracy', ascending=False)
    
    print("\n" + "="*80)
    print("GLOBAL SUMMARY")
    print("="*80)
    print(f"Total rules extracted: {len(all_rules_df)}")
    if len(all_rules_df) > 0:
        print(f"Best rule accuracy: {all_rules_df['test_accuracy'].max():.1%}")
        print(f"Mean rule accuracy: {all_rules_df['test_accuracy'].mean():.1%}")
        
        print("\n" + "="*80)
        print(f"TOP {min(NUMBER_OF_RULES, len(all_rules_df))} RULES")
        print("="*80)
        
        for idx, row in all_rules_df.head(NUMBER_OF_RULES).iterrows():
            print(f"\n{'='*70}")
            print(f"{row['target']} | Depth={row['depth']} | Tree={row['tree_id']} → {row['prediction']}")
            print(f"{'='*70}")
            print(f"  Train: {int(row['train_samples'])} samples, {row['train_confidence']:.1%} confidence")
            print(f"  Test:  {int(row['test_samples'])} samples ({row['test_pct']:.1f}%), {row['test_accuracy']:.1%} accuracy")
            print(f"  Conditions ({row['n_conditions']} total):")
            for i, (feature, op, threshold) in enumerate(row['conditions'], 1):
                if threshold == -999:
                    print(f"    {i}. {feature} {op} -999 (missing)")
                else:
                    print(f"    {i}. {feature} {op} {threshold:.6f}")
        
        # Summary by depth
        print("\n" + "="*80)
        print("BREAKDOWN BY DEPTH")
        print("="*80)
        depth_summary = all_rules_df.groupby('depth').agg({
            'test_accuracy': ['count', 'mean', 'max']
        }).round(4)
        print(depth_summary)
        
        # Summary by target
        print("\n" + "="*80)
        print("BREAKDOWN BY TARGET")
        print("="*80)
        target_summary = all_rules_df.groupby('target').agg({
            'test_accuracy': ['count', 'mean', 'max']
        }).round(4)
        print(target_summary)

print("\n✓ Ensemble rule extraction complete!")
print(f"✓ All rules stored in 'all_rules_df'")

In [15]:
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

# ===== PARAMETERS =====
N_TREES = 5
TREE_DEPTHS = [2, 3]
MIN_TEST_SAMPLES = 50
MIN_TEST_PCT = 1.0
NUMBER_OF_RULES = 300
POSITIVE_ONLY = True

print("="*80)
print("ENSEMBLE RULE EXTRACTION WITH TIME SERIES CV")
print("="*80)
print(f"Trees per target: {N_TREES}")
print(f"Tree depths: {TREE_DEPTHS}")
print(f"Filters: ≥{MIN_TEST_SAMPLES} samples, ≥{MIN_TEST_PCT}% coverage")
print(f"Show top: {NUMBER_OF_RULES} rules")
print(f"Positive only: {POSITIVE_ONLY}")

# Use the same features and filled data
X_all_filled = merged_common_final_selected[final_features].fillna(-999)

# Time series split for stability evaluation
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

def extract_rules(tree, feature_names):
    """Extract all decision rules from tree"""
    tree_ = tree.tree_
    feature_name = [feature_names[i] if i != _tree.TREE_UNDEFINED else "undefined!" 
                    for i in tree_.feature]
    
    paths = []
    
    def recurse(node, path):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name = feature_name[node]
            threshold = tree_.threshold[node]
            recurse(tree_.children_left[node], path + [(name, '<=', threshold)])
            recurse(tree_.children_right[node], path + [(name, '>', threshold)])
        else:
            samples = tree_.n_node_samples[node]
            weighted_values = tree_.value[node][0]
            class_pred = int(np.argmax(weighted_values))
            total_weight = weighted_values.sum()
            confidence = weighted_values[class_pred] / total_weight if total_weight > 0 else 0
            
            paths.append({
                'rule': path,
                'prediction': class_pred,
                'samples': samples,
                'confidence': confidence,
            })
    
    recurse(0, [])
    return paths

# Store all rules across all targets and depths
all_rules_global = []

for depth in TREE_DEPTHS:
    for target_name in target_cols:
        print(f"\nProcessing {target_name} (depth={depth})...", end=" ")
        
        y_all = merged_common_final_selected[target_name]
        
        # Calculate class weights on full data
        n_neg = (y_all == 0).sum()
        n_pos = (y_all == 1).sum()
        scale_pos_weight = n_neg / n_pos
        class_weight_dict = {0: 1.0, 1: scale_pos_weight}
        
        # Train multiple trees on FULL dataset
        for i in range(N_TREES):
            tree = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=100,
                min_samples_leaf=50,
                min_impurity_decrease=0.0001,
                max_features='sqrt',
                class_weight=class_weight_dict,
                random_state=42+i
            )
            
            # Bootstrap sample from FULL data
            sample_idx = np.random.RandomState(42+i).choice(
                len(X_all_filled), 
                size=int(len(X_all_filled)*0.8), 
                replace=True
            )
            X_sample = X_all_filled.iloc[sample_idx]
            y_sample = y_all.iloc[sample_idx]
            
            tree.fit(X_sample, y_sample)
            
            # Extract rules
            rules = extract_rules(tree, X_all_filled.columns.tolist())
            
            # Evaluate each rule on time series CV folds
            for rule_info in rules:
                conditions = rule_info['rule']
                
                fold_train_accs = []
                fold_test_accs = []
                fold_test_samples = []
                
                for train_idx, test_idx in tscv.split(X_all_filled):
                    X_train_fold = X_all_filled.iloc[train_idx]
                    X_test_fold = X_all_filled.iloc[test_idx]
                    y_train_fold = y_all.iloc[train_idx]
                    y_test_fold = y_all.iloc[test_idx]
                    
                    # Apply rule on train fold
                    train_mask = np.ones(len(X_train_fold), dtype=bool)
                    for feature, operator, threshold in conditions:
                        if operator == '<=':
                            train_mask &= (X_train_fold[feature] <= threshold).values
                        else:
                            train_mask &= (X_train_fold[feature] > threshold).values
                    
                    train_matched = train_mask.sum()
                    
                    # Apply rule on test fold
                    test_mask = np.ones(len(X_test_fold), dtype=bool)
                    for feature, operator, threshold in conditions:
                        if operator == '<=':
                            test_mask &= (X_test_fold[feature] <= threshold).values
                        else:
                            test_mask &= (X_test_fold[feature] > threshold).values
                    
                    test_matched = test_mask.sum()
                    
                    if train_matched > 0:
                        train_acc = (y_train_fold[train_mask] == rule_info['prediction']).mean()
                        fold_train_accs.append(train_acc)
                    
                    if test_matched > 0:
                        test_acc = (y_test_fold[test_mask] == rule_info['prediction']).mean()
                        fold_test_accs.append(test_acc)
                        fold_test_samples.append(test_matched)
                
                # Only keep rules that matched in at least 3 folds
                if len(fold_test_accs) >= 3:
                    avg_train_acc = np.mean(fold_train_accs) if fold_train_accs else 0
                    avg_test_acc = np.mean(fold_test_accs)
                    avg_test_samples = np.mean(fold_test_samples)
                    std_test_acc = np.std(fold_test_accs)
                    stability_gap = abs(avg_train_acc - avg_test_acc)
                    
                    # Apply filters
                    if avg_test_samples >= MIN_TEST_SAMPLES:
                        all_rules_global.append({
                            'target': target_name,
                            'depth': depth,
                            'tree_id': i,
                            'prediction': 'POSITIVE' if rule_info['prediction'] == 1 else 'NEGATIVE',
                            'train_samples': rule_info['samples'],
                            'train_confidence': rule_info['confidence'],
                            'avg_train_acc': avg_train_acc,
                            'avg_test_acc': avg_test_acc,
                            'std_test_acc': std_test_acc,
                            'avg_test_samples': avg_test_samples,
                            'stability_gap': stability_gap,
                            'n_folds_matched': len(fold_test_accs),
                            'n_conditions': len(conditions),
                            'conditions': conditions
                        })
        
        print(f"✓")

# Create DataFrame
all_rules_df = pd.DataFrame(all_rules_global)

if len(all_rules_df) == 0:
    print("\n⚠ NO RULES FOUND - try lowering MIN_TEST_SAMPLES or MIN_TEST_PCT")
else:
    # Filter by POSITIVE_ONLY if enabled
    if POSITIVE_ONLY:
        all_rules_df = all_rules_df[all_rules_df['prediction'] == 'POSITIVE']
    
    print("\n" + "="*80)
    print("GLOBAL SUMMARY")
    print("="*80)
    print(f"Total rules extracted: {len(all_rules_df)}")
    if len(all_rules_df) > 0:
        print(f"Best rule test accuracy: {all_rules_df['avg_test_acc'].max():.1%}")
        print(f"Mean rule test accuracy: {all_rules_df['avg_test_acc'].mean():.1%}")
        print(f"Mean stability gap: {all_rules_df['stability_gap'].mean():.1%}")
        print(f"Best stability (lowest gap): {all_rules_df['stability_gap'].min():.1%}")
        
        # Sort by test accuracy
        print("\n" + "="*80)
        print(f"TOP {min(NUMBER_OF_RULES, len(all_rules_df))} RULES (BY TEST ACCURACY)")
        print("="*80)
        
        all_rules_df_sorted = all_rules_df.sort_values('avg_test_acc', ascending=False)
        
        for idx, row in all_rules_df_sorted.head(NUMBER_OF_RULES).iterrows():
            stability_flag = "✓ STABLE" if row['stability_gap'] < 0.10 else "⚠️ UNSTABLE"
            print(f"\n{'='*70}")
            print(f"{row['target']} | Depth={row['depth']} | Tree={row['tree_id']} → {row['prediction']}")
            print(f"{'='*70}")
            print(f"  Train: {row['avg_train_acc']:.1%} accuracy")
            print(f"  Test:  {row['avg_test_acc']:.1%} ± {row['std_test_acc']:.1%} accuracy ({row['avg_test_samples']:.0f} samples)")
            print(f"  Stability: Gap={row['stability_gap']:.1%} {stability_flag} | Matched in {row['n_folds_matched']}/5 folds")
            print(f"  Conditions ({row['n_conditions']} total):")
            for i, (feature, op, threshold) in enumerate(row['conditions'], 1):
                if threshold == -999:
                    print(f"    {i}. {feature} {op} -999 (missing)")
                else:
                    print(f"    {i}. {feature} {op} {threshold:.6f}")
        
        # Sort by stability (lowest gap)
        print("\n" + "="*80)
        print(f"TOP {min(50, len(all_rules_df))} MOST STABLE RULES (BY TRAIN-TEST GAP)")
        print("="*80)
        
        all_rules_df_stable = all_rules_df.sort_values('stability_gap', ascending=True)
        
        for idx, row in all_rules_df_stable.head(50).iterrows():
            print(f"\n{'='*70}")
            print(f"{row['target']} | Depth={row['depth']} | Tree={row['tree_id']} → {row['prediction']}")
            print(f"{'='*70}")
            print(f"  Train: {row['avg_train_acc']:.1%} | Test: {row['avg_test_acc']:.1%} ± {row['std_test_acc']:.1%}")
            print(f"  Stability: Gap={row['stability_gap']:.1%} ✓ ({row['n_folds_matched']}/5 folds, {row['avg_test_samples']:.0f} samples)")
            print(f"  Conditions ({row['n_conditions']} total):")
            for i, (feature, op, threshold) in enumerate(row['conditions'], 1):
                if threshold == -999:
                    print(f"    {i}. {feature} {op} -999 (missing)")
                else:
                    print(f"    {i}. {feature} {op} {threshold:.6f}")
        
        # Summary statistics
        print("\n" + "="*80)
        print("BREAKDOWN BY DEPTH")
        print("="*80)
        depth_summary = all_rules_df.groupby('depth').agg({
            'avg_test_acc': ['count', 'mean', 'max'],
            'stability_gap': 'mean'
        }).round(4)
        print(depth_summary)
        
        print("\n" + "="*80)
        print("BREAKDOWN BY TARGET")
        print("="*80)
        target_summary = all_rules_df.groupby('target').agg({
            'avg_test_acc': ['count', 'mean', 'max'],
            'stability_gap': 'mean'
        }).round(4)
        print(target_summary)

print("\n✓ Ensemble rule extraction complete!")
print(f"✓ All rules stored in 'all_rules_df'")